# AM Configuration File Generation and Analysis

This notebook prepares atmospheric model inputs, builds AM configuration files, runs related processing steps, and visualizes transmittance and site-level comparison products.

## How this notebook is organized

This cleaned version preserves the original computational workflow, but restructures it into a more readable sequence. Each code block is preceded by a short markdown explanation so another reader can understand the purpose of the step before executing it.

## Notes for reuse

- Confirm that all referenced data files are available at the paths used in the code.
- Run cells from top to bottom because later sections depend on variables defined earlier.
- Review hard-coded site names, coordinates, and output paths before adapting the notebook for a different project.

## Project setup and imports

### Step 1

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
import matplotlib.pyplot as plt 
import matplotlib as mpl
import pandas as pd
import collections
import numpy as np
import glob
import matplotlib.pyplot as plt
# from mpl_toolkits.basemap import Basemap
import re
from pathlib import Path
from scipy.interpolate import griddata
from scipy.interpolate import interp2d
from scipy.stats import gaussian_kde
from scipy.stats import spearmanr
import xarray as xr
import pandas as pd
import glob
# import seaborn as sns
# from sklearn.metrics import mean_squared_error
import os
import xarray
import subprocess


## Pressure and PWV calculations

### Step 2

This cell defines reusable helper function(s) `calculate_pressure` so later sections can apply the same processing logic consistently.

In [ ]:


def calculate_pressure(P_b, T_b, L_b, h, h_b, R_star=8.3144598, g_0=9.80665, M=0.028964425278793993):
    T_h = T_b - L_b * (h - h_b)
    exponent = (g_0 * M) / (R_star * L_b)
    pressure = P_b * (T_h / T_b) ** exponent
    return pressure

# Example usage:
P_b = 101325  # reference pressure at sea level in Pa
T_b = 288.15  # reference temperature at sea level in K
L_b = 0.0065  # temperature lapse rate in K/m
h = 4500     # height at which pressure is calculated in m
h_b = 00       # height of reference level in m

pressure = calculate_pressure(P_b, T_b, L_b, h, h_b)
print(f"Pressure at {h} meters: {pressure:.2f} Pa")


## Interpolation and extraction

### Step 3

This cell loads dataset(s) `era_5_am_new.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
era5 = xr.open_dataset('era_5_am_new.nc')
era5 = era5.rename({'valid_time': 'time'})
era5 = era5.rename({'pressure_level': 'level'})
era5


## Data loading and inspection

### Step 4

This cell loads dataset(s) `era_5_am.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
# era5 = xr.open_dataset('era_5_am.nc')


## Analysis workflow

### Step 5

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
# era5


## Site definitions and metadata

### Step 6

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
# concerde points for Ladakh region.

HANLE = { 'name' : 'Hanle',
         'lat' : 32.7789,
         'lon' : 78.9650,
         'elevation' : 4500,}

MERAK = { 'name' : 'Merak',
            'lat' : 33.7828,
            'lon' : 78.5778,
            'elevation' : 4310,}

SITE_A = { 'name' : 'Site A',
            'lat' : 34.25,
            'lon' : 78.75,
            'elevation' : 4800,}


SITE_B = { 'name' : 'Site B',
            'lat' : 32.5,
            'lon' : 79,
            'elevation' : 4500,}


### Step 7

This cell defines reusable helper function(s) `extract_data`, `compute_statistics` so later sections can apply the same processing logic consistently.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# Load ERA5 data
era5 = xr.open_dataset('era_5_am.nc')
#era5 = era5.sel(time=slice('2019-01-01', '2019-01-31'))

# Define points of interest
points_of_interest = {
    'Hanle': {'lat': 32.7789, 'lon': 78.9650, 'elevation': 4500},
    'Merak': {'lat': 33.7828, 'lon': 78.5778, 'elevation': 4310},
    'Site A': {'lat': 34.25, 'lon': 78.75, 'elevation': 4800},
    'Site B':   {'lat': 32.5, 'lon': 79, 'elevation': 4500}
}

# Filter for winter months (December, January, February)
winter_months = [1, 2]
#winter_months = [1,2,3,4,5,6,7,8,9,10,11,12]
era5_winter = era5.where(era5['time.month'].isin(winter_months), drop=True)

# Function to extract data for a given point
def extract_data(point):
    lat = point['lat']
    lon = point['lon']
    data = era5_winter.interp(latitude=lat, longitude=lon)
    #data = era5_winter.sel(latitude=lat, longitude=lon, method='nearest')
    return data

# Function to compute median, 25th and 75th percentiles at each pressure level
def compute_statistics(data, var):
    median_values = data[var].median(dim='time')
    p25_values = data[var].quantile(0.05, dim='time')
    p75_values = data[var].quantile(0.95, dim='time')
    return median_values, p25_values, p75_values

# Setup plot with desired aesthetics
plt.rc('font', family='serif', size=16)
plt.rcParams['xtick.major.size'] = 6
plt.rcParams['xtick.minor.size'] = 4
plt.rcParams['ytick.major.size'] = 6
plt.rcParams['ytick.minor.size'] = 4

# Initialize subplots
fig, (ax_temp, ax_humidity) = plt.subplots(1, 2, figsize=(11, 9), sharey=True)
#plt.subplots_adjust(wspace=0)

# Set wspace to 0 and manually adjust the subplot positions
plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1, wspace=0, hspace=0)

# Additional adjustments to ensure no space between plots
ax_temp.set_position([0.1, 0.1, 0.4, 0.8])
ax_humidity.set_position([0.5, 0.1, 0.4, 0.8])


# Process data for each point of interest
markers = ['o', 's', '^', 'D']  # Different markers for different sites
line_styles = ['-', '--', ':', '-.']  # Different line styles for different sites
line_colors = ['blue', 'red', 'green', 'cyan']  # Different line colors for different sites
for (point_name, point), marker,line_styles,line_colors in zip(points_of_interest.items(), markers,line_styles, line_colors):
    data = extract_data(point)
    temp_median, temp_p25, temp_p75 = compute_statistics(data, 't')  # Temperature
    humidity_median, humidity_p25, humidity_p75 = compute_statistics(data, 'q')  # Specific Humidity
    pressure_levels = data['level']

    # Plot Temperature vs Pressure
    ax_temp.plot(temp_median, pressure_levels, label=f'{point_name}', marker=marker, linestyle='-', markersize=10, markerfacecolor='white', markeredgewidth=3,
                 alpha = 0.5)
    ax_temp.fill_betweenx(pressure_levels, temp_p25, temp_p75, alpha=0.3)
    
    # Plot Specific Humidity vs Pressure
    ax_humidity.plot(humidity_median, pressure_levels, marker=marker, linestyle='-', markersize=10, markerfacecolor='white', markeredgewidth=3,
                     alpha = 0.5)
    ax_humidity.fill_betweenx(pressure_levels, humidity_p25, humidity_p75, alpha=0.3)
    
    # Plot horizontal lines representing site elevations
    print(point['elevation'])
    # calculate pressure level for the given elevation
    p = calculate_pressure(101325, 288.15, 0.0065, point['elevation'], 0)/100
    print(p)
    #ax_temp.axhline(y=p, color='gray', linestyle=line_styles, linewidth=2,c = line_colors,
             #alpha = 0.7)
    ax_temp.axhline( y = 55110.85655074304/100, color = 'gray', linestyle = '--', linewidth = 2, alpha = 0.7)
    ax_temp.axvline( x = 252.97362870704214, color = 'gray', linestyle = '--', linewidth = 2, alpha = 0.7)
    ax_humidity.axhline(y=p, color='gray', linestyle=line_styles, linewidth=2,c = line_colors,
                label=f'{point_name} elevation', alpha = 0.7)

# Customize Temperature plot
ax_temp.set_ylim(1000, -20)  # Invert y-axis to start from 1000 hPa to 1 hPa
ax_temp.set_xlabel('Temperature (K)', fontsize=20)
ax_temp.set_ylabel('Pressure (hPa)', fontsize=20)
ax_temp.legend(loc='best', fontsize=16, frameon=True, facecolor='white', edgecolor='black', framealpha=1, shadow=True)

# Customize Specific Humidity plot
ax_humidity.set_ylim(1000, -20)  # Invert y-axis to start from 1000 hPa to 1 hPa
ax_humidity.set_xlabel(r'Specific Humidity (kg/kg $\times 10^{-4}$)', fontsize=20)
ax_humidity.xaxis.get_offset_text().set_visible(False)
ax_humidity.ticklabel_format(axis='x', style='sci', scilimits=(0, 0))
ax_humidity.legend(loc='upper right', fontsize=16, frameon=True, facecolor='white', edgecolor='black', framealpha=1, shadow=True)

# Apply formatting to axes and plot borders
for ax in [ax_temp, ax_humidity]:
    ax.set_facecolor('white')
    ax.spines['top'].set_linewidth(2)
    ax.spines['right'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)
    ax.spines['left'].set_linewidth(2)
    ax.tick_params(axis='both', direction='in', width=1.5, labelsize=18, which='both')
    ax.minorticks_on()

# Adjust layout and show plots
#plt.tight_layout()
plt.savefig('Pressure_vs_Temperature_and_Humidity_Profiles.pdf', dpi=600)
plt.show()


### Step 8

This cell subsets the dataset to a selected time, level, region, or site so the next step works with a focused slice of the data.

In [ ]:
# Find the temperature at 1000 hPa for Hanle
data = extract_data(points_of_interest['Hanle'])
temp_1000_hPa = data['t'].sel(level=1000, method='nearest').mean(dim='time').values.item()
print(f"Temperature at 1000 hPa: {temp_1000_hPa:.2f} K")

data = extract_data(points_of_interest['Site A'])
temp_1000_hPa = data['t'].sel(level=1000, method='nearest').mean(dim='time').values.item()
print(f"Temperature at 1000 hPa: {temp_1000_hPa:.2f} K")

data = extract_data(points_of_interest['Site B'])
temp_1000_hPa = data['t'].sel(level=1000, method='nearest').mean(dim='time').values.item()
print(f"Temperature at 1000 hPa: {temp_1000_hPa:.2f} K")

# data = extract_data(points_of_interest['Site X'])
# temp_1000_hPa = data['t'].sel(level=1000, method='nearest').mean(dim='time').values.item()
# print(f"Temperature at 1000 hPa: {temp_1000_hPa:.2f} K")


### Step 9

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
# exract the whole data of Hanle and save in sites/hanle/hanle_data_100.nc 
data = era5.interp(latitude=HANLE['lat'], longitude=HANLE['lon'], method='cubic')
# filter the data for winter months (November, December, January, February)
data = data.where(data['time.month'].isin([1,2,11,12]), drop=True)
data.to_netcdf('sites/hanle/hanle_data_1000.nc')

data = era5.interp(latitude=MERAK['lat'], longitude=MERAK['lon'], method='cubic')
data = data.where(data['time.month'].isin([1,2,11,12]), drop=True)
data.to_netcdf('sites/merak/merak_data_1000.nc')

data = era5.interp(latitude=SITE_A['lat'], longitude=SITE_A['lon'], method='cubic')
data = data.where(data['time.month'].isin([1,2,11,12]), drop=True)
data.to_netcdf('sites/site_a/site_a_data_1000.nc')

data = era5.interp(latitude=SITE_B['lat'], longitude=SITE_B['lon'], method='cubic')
data = data.where(data['time.month'].isin([1,2,11,12]), drop=True)
data.to_netcdf('sites/site_b/site_b_data_1000.nc')


### Step 10

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
data = era5.interp(latitude=HANLE['lat'], longitude=HANLE['lon'], method='cubic')
data.to_netcdf('sites/hanle/hanle_data_100.nc')

data = era5.interp(latitude=MERAK['lat'], longitude=MERAK['lon'], method='cubic')
data.to_netcdf('sites/merak/merak_data_100.nc')

data = era5.interp(latitude=SITE_A['lat'], longitude=SITE_A['lon'], method='cubic')
data.to_netcdf('sites/site_a/site_a_data_100.nc')

data = era5.interp(latitude=SITE_B['lat'], longitude=SITE_B['lon'], method='cubic')
data.to_netcdf('sites/site_b/site_b_data_100.nc')


### Step 11

This cell loads dataset(s) `site_a_data_1000.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
hanle_wint = xr.open_dataset('sites/site_a/site_a_data_1000.nc')
print(hanle_wint)
# print the time dimension of the data
print(hanle_wint.time)


## Data loading and inspection

### Step 12

This cell loads dataset(s) `era5_ladakh_all.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
era_5_all = xr.open_dataset('era5_ladakh_all.nc')


### Step 13

This cell loads dataset(s) `pressure.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
pressure_data = xarray.open_dataset('pressure.nc')
print(pressure_data)


## Mapping and visualization

### Step 14

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
surface_pressure = pressure_data['sp']
hanle = {'lat': 32.7789, 'lon': 78.9650}
#hanle = {'lat': 19.823, 'lon': 204.5306}
surface_pressure_hanle = surface_pressure.sel(latitude=hanle['lat'], longitude=hanle['lon'], method='nearest')
surface_pressure_hanle = surface_pressure_hanle.sel(time=slice('2010-01-01', '2017-12-31'))


surface_pressure_hanle_interp = surface_pressure.interp(latitude = hanle['lat'], longitude = hanle['lon'],
        )
print(surface_pressure_hanle_interp)
surface_pressure_hanle_interp = surface_pressure_hanle_interp.sel(time=slice('2010-01-01', '2017-12-31'))



t2_m = pressure_data['t2m']
t2m_hanle = t2_m.sel(latitude=hanle['lat'], longitude=hanle['lon'], method='nearest')
t2m_hanle = t2m_hanle.sel(time=slice('2010-01-01', '2017-12-31'))

t2m_hanle_interp = t2_m.interp(latitude = hanle['lat'], longitude = hanle['lon'])
t2m_hanle_interp = t2m_hanle_interp.sel(time=slice('2010-01-01', '2017-12-31'))

plt.subplot(2, 1, 1)
surface_pressure_hanle.plot(label='original')
surface_pressure_hanle_interp.plot(label='interpolated' )
plt.legend()
plt.subplot(2, 1, 2)
t2m_hanle.plot(label='original')
t2m_hanle_interp.plot(label='interpolated')
plt.legend()
plt.show()




# resample the data to monthly data (take median of the month) accross the years
surface_pressure_hanle_monthly = surface_pressure_hanle.groupby('time.month').median('time')
t2m_hanle_monthly = t2m_hanle.groupby('time.month').median('time')

surface_pressure_hanle_interp_monthly = surface_pressure_hanle_interp.groupby('time.month').median('time')
t2m_hanle_interp_monthly = t2m_hanle_interp.groupby('time.month').median('time')

plt.subplot(2, 1, 1)
surface_pressure_hanle_monthly.plot(label='original')
surface_pressure_hanle_interp_monthly.plot(label='interpolated')
plt.legend()
plt.subplot(2, 1, 2)
t2m_hanle_monthly.plot(label='original')
t2m_hanle_interp_monthly.plot(label='interpolated')
plt.legend()
plt.show()


## Pressure and PWV calculations

### Step 15

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
# calculate_pressure(101325, 273.15+13.6, 0.0065, 4500, 0)
calculate_pressure(101325, 280, 0.0065, 4929.86, 0)


## Mapping and visualization

### Step 16

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
t = [-10.4,-8.5,-4.7,0.3,4.8,
     9.6,13.6,12.9,8.7,1.8,-3.1,-7.4]
p = [584.6,585.0,587.8,589.7,590.1,589.5,589.8,590.8,591.4,591.4,589.8,587.3]
x = [0,1,2,3,4,5,6,7,8,9,10,11]
t_n = [-12.4,-10.5,-7.2,-2.4,2.1,7.0,10.9,10.1,6.0,-1.0,-5.7,-9.3]
p_n = [584.7,585.0,587.8,589.7,590.1,589.5,589.8,590.8,591.4,591.4,589.8,587.3]

t_era5 = t2m_hanle_interp_monthly.values
p_era5_calculated = []
for i in range(len(t_era5)):
    p_era5_calculated.append(calculate_pressure(101325,t_era5[i], 0.0065, 4500, 0)/100)


calculated_p = []
calculated_p_n = []
for i in range(len(t)):
    calculated_p.append(calculate_pressure(101325, 273.15+t[i], 0.0065, 4500, 0)/100)
    calculated_p_n.append(calculate_pressure(101325, 273.15+t_n[i], 0.0065, 4500, 0)/100)


plt.plot(x, p, label='Observed: IAO (D)', color='red',marker='o',alpha=0.5)
plt.plot(x, p_n, label='Observed: IAO (N)', color='black',marker='X',alpha=0.4)
#plt.plot(x, calculated_p, label='Calculated from Actual Temp Data (D)', color='blue',marker='x')
#plt.plot(x,calculated_p_n, label='Calculated from Actual Temp Data (N)', color='orange',marker='d')
plt.plot(x,surface_pressure_hanle_interp_monthly.values/100, label='Directly from ERA5', color='green',marker='s')
plt.plot(x,p_era5_calculated, label='Calculated from ERA5 Temp Data', color='purple',marker='p')

plt.legend(fontsize=8)
plt.ylabel('Pressure (hPa)')
plt.xlabel('Month')
plt.tight_layout()
plt.savefig('pressure_comparison_new.pdf', dpi=600)
plt.show()


## Making Config files

## Analysis workflow

### Step 17

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
# fetch the levels in the data
levels = data['level'].values
print(levels)
levels = [int(level) for level in levels]
sorted_levels = sorted(levels)
print(sorted_levels)
levels = sorted_levels
# sort the levels in ascending order
# levels = np.sort(levels)
# print(levels)


## Exports and file generation

### Step 18

This cell writes derived results or configuration content to disk so they can be reused outside the notebook.

In [ ]:
# save the levels in a am_level.txt file in separate lines
with open('am_levels.txt', 'w') as f:
    for item in levels:
        f.write("%s\n" % item)


## Helper functions

### Step 19

This cell defines reusable helper function(s) `O3_mmr_to_vmr`, `QV_to_H2O` so later sections can apply the same processing logic consistently.

In [ ]:
def O3_mmr_to_vmr(o3_mmr):                                                  #O3 mmr to vmr conversion
    o3_vmr = 1e-6*(1e6 * (28.964 / 47.9982) * o3_mmr)
    return o3_vmr
def QV_to_H2O(qv_mmr):                                                      #QV mmr to vmr conversion                    
    h2o = qv_mmr/0.622
    return h2o


### Step 20

This cell defines reusable helper function(s) `Layername` so later sections can apply the same processing logic consistently.

In [ ]:
def Layername(pressure):                                                    #Layer name definition (for am)
    if pressure < 1:
        return 'mesosphere'
    elif pressure >= 1 and pressure < 100:
        return 'stratosphere'
    else:
        return 'troposphere'


## Pressure and PWV calculations

### Step 21

This cell defines reusable helper function(s) `Layercreation` so later sections can apply the same processing logic consistently.

In [ ]:
def Layercreation(line):
    layer = {}
    layer['Name'] = Layername(line[0])
    layer['Pressure'] = str(line[0])
    layer['Temperature'] = str(round(line[1],2))
    if line[0] <= 1:
        layer['Lineshape'] = 'Voigt-Kielkopf'
    layer['Ozone'] = '{:.3e}'.format(line[2])
    layer['PWV'] = '{:.3e}'.format(line[3])
    return layer


### Step 22

This cell defines reusable helper function(s) `FileWriting` so later sections can apply the same processing logic consistently.

In [ ]:
def FileWriting(data, filename, pressure, preamble, keyword):        
    with open('sites/'+keyword+'/am/'+filename+'.amc', 'w') as am_file:
        am_file.write(''.join(preamble)+'\n')
        for n in range(data.shape[0]):
            line = [pressure[n]] + data[n].tolist()
            layer = Layercreation(line)
            section = ['layer '+layer['Name']]
            section += ['Pbase '+layer['Pressure']+' mbar']
            section += ['Tbase '+layer['Temperature']+' K']
            if 'Lineshape' in layer:
                section += ['lineshape '+layer['Lineshape']]
            section += ['column dry_air vmr']
            section += ['column h2o vmr '+layer['PWV']]
            section += ['column o3 vmr '+ layer['Ozone']]
            am_file.write('\n'.join(section)+'\n\n')
    print('\rFile written: '+filename+'.amc', end='\r', flush=True)


## Site definitions and metadata

### Step 23

This cell defines reusable helper function(s) `Metadata` so later sections can apply the same processing logic consistently.

In [ ]:
def Metadata(keyword=None):
    if keyword == None:
        keyword = input('Enter location keyword: ')
    metadata = {}
    with open('program/metadata/'+keyword+'_metadata.txt') as f:
        for line in f:
            (key, val) = line.split('=')
            if '\n' in val:
                val = val[:-1]
            metadata[key] = val
    metadata['COORDINATES'] = tuple(map(float, metadata['COORDINATES'].split(',')))
    metadata['SURFACEPRES'] = float(metadata['SURFACEPRES'])
    if not metadata['SHORTNAME'] in os.listdir('sites'):
        os.mkdir('sites/'+metadata['SHORTNAME'])
        print('\nDirectory created:',metadata['SHORTNAME'])
    print('\nSite:\n'+metadata['NAME'], end='\n\n')
    return metadata


### Step 24

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:
metadata = Metadata()


## Exports and file generation

### Step 25

This cell loads one or more datasets into memory for later processing.

In [ ]:
flairs = ['o3', 't', 'q']
quarters = ['0.25','0.5','0.75']
preamble = open('program/preamble.txt', 'r').readlines()
metadata = Metadata()
key_num = input('Define number of data points: ')
if 'am' not in os.listdir('sites/'+metadata['SHORTNAME']):
    os.mkdir(f'sites/{metadata["SHORTNAME"]}/am')
if 'am_output' not in os.listdir('sites/'+metadata['SHORTNAME']):
    os.mkdir(f'sites/{metadata["SHORTNAME"]}/am_output')
 
dataset_interp = xarray.open_dataset(f'sites/{"/".join((metadata["SHORTNAME"],metadata["SHORTNAME"]))}_data_{key_num}.nc')  
levs = np.array([i.data for i in dataset_interp.variables['level'][:]])
am = os.listdir('sites/'+metadata['SHORTNAME']+'/am')
start = metadata['STARTFREQ']+' '+metadata['MEASURE']
end = metadata['ENDFREQ']+' '+metadata['MEASURE']
df = metadata['STEPFREQ']+' '+metadata['MEASURE']
zenith = metadata['ZENITH']+' deg'
Nscale = metadata['NSCALE']
command = ' '.join([start,end,df,zenith,Nscale])

pressure = dataset_interp.variables['level'][:].data
percentiles = {}
for quarter in quarters:
    to_dict = {}
    for flair in flairs:
        data_mean = []
        for i in range(levs.size):
            data = np.array(dataset_interp.variables[flair][:,i])
            p = np.linspace(0,1,data.size)
            data.sort()
            f = lambda x : np.interp(x, p, data)
            data_mean.append(f(float(quarter)))
        to_dict[flair] = np.array(data_mean)
    percentiles[quarter] = to_dict
for q in np.arange(0.25,1,0.25):
    to_sort = np.array([percentiles[str(q)][value] for value in ['t', 'o3', 'q']])
    to_sort = np.array([to_sort[0], O3_mmr_to_vmr(to_sort[1]), QV_to_H2O(to_sort[2])])
    to_file = np.array([[i[j] for i in to_sort] for j in range(0,to_sort.shape[1])])
    filename = f'{metadata["SHORTNAME"]}_{key_num}_{str(int(q*100))}'
    FileWriting(to_file, filename, pressure, preamble, metadata['SHORTNAME'])

am = np.array(os.listdir('sites/'+metadata['SHORTNAME']+'/am'))[[metadata['SHORTNAME'] in a for a in np.array(os.listdir('sites/'+metadata['SHORTNAME']+'/am'))]]
am = am[[key_num in i for i in am]]
for file in am:
    print('\rFile to model: '+file, end='\r', flush=True)
    am_input = 'am sites/'+metadata['SHORTNAME']+'/am/'+file+' '+command
    values=subprocess.run(am_input, stdout=subprocess.PIPE,
                          stderr=subprocess.PIPE,
                          universal_newlines=True, shell=True)
    with open('sites/'+metadata['SHORTNAME']+'/am_output/'+file[:-4]+'.out', 'w') as txt_file:
        txt_file.write(values.stdout)
    with open('sites/'+metadata['SHORTNAME']+'/am_output/'+file[:-4]+'_stderr.out', 'w') as txt_file:
        txt_file.write(values.stderr)


## Mapping and visualization

### Step 26

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
# from sites/hanle/am_output/
# extract the output files and plot the data
data_25 = np.loadtxt('sites/hanle/am_output/hanle_100_25.out')
data_50 = np.loadtxt('sites/hanle/am_output/hanle_100_50.out')
data_75 = np.loadtxt('sites/hanle/am_output/hanle_100_75.out')

plt.figure(figsize=(10, 6))
#plt.plot(data_25[:, 0], data_25[:, 2], label='25th percentile', color='blue')
plt.plot(data_25[:, 0], data_25[:, 3], label='25th percentile', color='blue')
# plt.plot(data_50[:, 0], data_50[:, 2], label='50th percentile', color='red')
plt.plot(data_50[:, 0], data_50[:, 3], label='50th percentile', color='red')
# plt.plot(data_75[:, 0], data_75[:, 2], label='75th percentile', color='green')
plt.plot(data_75[:, 0], data_75[:, 3], label='75th percentile', color='green')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Transmittance')
plt.legend()
plt.title('Transmittance vs Wavelength')
#plt.grid()
#plt.savefig('Transmittance_vs_Wavelength.pdf', dpi=600)
plt.show()

# count the number of elements in the data
print(len(data_25[:, 0]))
print(len(data_50[:, 0]))
print(len(data_75[:, 0]))


### Step 27

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
# load all the data to plot in a 2*2 subplot format for all the sites

fig, axs = plt.subplots(2, 2, figsize=(15, 10))

# load the data for Hanle
data_25 = np.loadtxt('sites/hanle/am_output/hanle_100_25.out')
data_50 = np.loadtxt('sites/hanle/am_output/hanle_100_50.out')
data_75 = np.loadtxt('sites/hanle/am_output/hanle_100_75.out')

axs[0, 0].plot(data_25[:, 0], data_25[:, 2], label='25th percentile', color='blue')
axs[0, 0].plot(data_50[:, 0], data_50[:, 2], label='50th percentile', color='red')
axs[0, 0].plot(data_75[:, 0], data_75[:, 2], label='75th percentile', color='green')
axs[0, 0].set_title('Hanle')
axs[0, 0].set_xlabel('Wavelength (nm)')
axs[0, 0].set_ylabel('Transmittance')
axs[0, 0].legend()

# load the data for Merak
data_25 = np.loadtxt('sites/merak/am_output/merak_100_25.out')
data_50 = np.loadtxt('sites/merak/am_output/merak_100_50.out')
data_75 = np.loadtxt('sites/merak/am_output/merak_100_75.out')

axs[0, 1].plot(data_25[:, 0], data_25[:, 2], label='25th percentile', color='blue')
axs[0, 1].plot(data_50[:, 0], data_50[:, 2], label='50th percentile', color='red')
axs[0, 1].plot(data_75[:, 0], data_75[:, 2], label='75th percentile', color='green')
axs[0, 1].set_title('Merak')
axs[0, 1].set_xlabel('Wavelength (nm)')
axs[0, 1].set_ylabel('Transmittance')
axs[0, 1].legend()


# load the data for Site A
data_25 = np.loadtxt('sites/site_a/am_output/site_a_100_25.out')
data_50 = np.loadtxt('sites/site_a/am_output/site_a_100_50.out')
data_75 = np.loadtxt('sites/site_a/am_output/site_a_100_75.out')

axs[1, 0].plot(data_25[:, 0], data_25[:, 2], label='25th percentile', color='blue')
axs[1, 0].plot(data_50[:, 0], data_50[:, 2], label='50th percentile', color='red')
axs[1, 0].plot(data_75[:, 0], data_75[:, 2], label='75th percentile', color='green')
axs[1, 0].set_title('Site A')
axs[1, 0].set_xlabel('Wavelength (nm)')
axs[1, 0].set_ylabel('Transmittance')
axs[1, 0].legend()

# load the data for Site B
data_25 = np.loadtxt('sites/site_b/am_output/site_b_100_25.out')
data_50 = np.loadtxt('sites/site_b/am_output/site_b_100_50.out')
data_75 = np.loadtxt('sites/site_b/am_output/site_b_100_75.out')

axs[1, 1].plot(data_25[:, 0], data_25[:, 2], label='25th percentile', color='blue')
axs[1, 1].plot(data_50[:, 0], data_50[:, 2], label='50th percentile', color='red')
axs[1, 1].plot(data_75[:, 0], data_75[:, 2], label='75th percentile', color='green')
axs[1, 1].set_title('Site B')
axs[1, 1].set_xlabel('Wavelength (nm)')
axs[1, 1].set_ylabel('Transmittance')
axs[1, 1].legend()

plt.tight_layout()
#plt.savefig('Transmittance_vs_Wavelength_all_sites.pdf', dpi=600)
plt.show()


### Step 28

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
# now plot three plots with 4 25,50,75 percentile data for all the sites in a single plot each

plt.figure(figsize=(10, 6))

# load the data for Hanle
hanle_25 = np.loadtxt('sites/hanle/am_output/hanle_100_25.out')
merak_25 = np.loadtxt('sites/merak/am_output/merak_100_25.out')
site_a_25 = np.loadtxt('sites/site_a/am_output/site_a_100_25.out')
site_b_25 = np.loadtxt('sites/site_b/am_output/site_b_100_25.out')

plt.plot(hanle_25[:, 0], hanle_25[:, 2], label='Hanle 25th percentile', color='blue')
plt.plot(merak_25[:, 0], merak_25[:, 2], label='Merak 25th percentile', color='red')
plt.plot(site_a_25[:, 0], site_a_25[:, 2], label='Site A 25th percentile', color='green')
plt.plot(site_b_25[:, 0], site_b_25[:, 2], label='Site B 25th percentile', color='orange')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Transmittance')
plt.legend()
plt.title('Transmittance vs Wavelength 25th percentile')

plt.tight_layout()
#plt.savefig('Transmittance_vs_Wavelength_25th_percentile.pdf', dpi=600)
plt.show()

plt.figure(figsize=(10, 6))

# load the data for Hanle
hanle_50 = np.loadtxt('sites/hanle/am_output/hanle_100_50.out')
merak_50 = np.loadtxt('sites/merak/am_output/merak_100_50.out')
site_a_50 = np.loadtxt('sites/site_a/am_output/site_a_100_50.out')
site_b_50 = np.loadtxt('sites/site_b/am_output/site_b_100_50.out')

plt.plot(hanle_50[:, 0], hanle_50[:, 2], label='Hanle 50th percentile', color='blue')
plt.plot(merak_50[:, 0], merak_50[:, 2], label='Merak 50th percentile', color='red')
plt.plot(site_a_50[:, 0], site_a_50[:, 2], label='Site A 50th percentile', color='green')
plt.plot(site_b_50[:, 0], site_b_50[:, 2], label='Site B 50th percentile', color='orange')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Transmittance')
plt.legend()
plt.title('Transmittance vs Wavelength 50th percentile')

plt.tight_layout()
#plt.savefig('Transmittance_vs_Wavelength_50th_percentile.pdf', dpi=600)
plt.show()

plt.figure(figsize=(10, 6))

# load the data for Hanle

hanle_75 = np.loadtxt('sites/hanle/am_output/hanle_100_75.out')
merak_75 = np.loadtxt('sites/merak/am_output/merak_100_75.out')
site_a_75 = np.loadtxt('sites/site_a/am_output/site_a_100_75.out')
site_b_75 = np.loadtxt('sites/site_b/am_output/site_b_100_75.out')

plt.plot(hanle_75[:, 0], hanle_75[:, 2], label='Hanle 75th percentile', color='blue')
plt.plot(merak_75[:, 0], merak_75[:, 2], label='Merak 75th percentile', color='red')
plt.plot(site_a_75[:, 0], site_a_75[:, 2], label='Site A 75th percentile', color='green')
plt.plot(site_b_75[:, 0], site_b_75[:, 2], label='Site B 75th percentile', color='orange')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Transmittance')
plt.legend()
plt.title('Transmittance vs Wavelength 75th percentile')
plt.tight_layout()

#plt.savefig('Transmittance_vs_Wavelength_75th_percentile.pdf', dpi=600)
plt.show()


## Site definitions and metadata

### Step 29

This cell loads dataset(s) `hanle_data_100.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
# load hanle_data_100.nc
hanle_data = xr.open_dataset('sites/hanle/hanle_data_100.nc')
print(hanle_data)


## Data loading and inspection

### Step 30

This cell loads dataset(s) `pressure_all.nc`, `o3_q_t_ladakh.nc` into memory so the notebook can inspect, subset, and analyze the underlying atmospheric variables.

In [ ]:
# load the pressure data
surface_pressure_data = xr.open_dataset('pressure_all.nc')

# load the q,t and o3 data
ladakh_data = xr.open_dataset('o3_q_t_ladakh.nc')


## Analysis workflow

### Step 31

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
ladakh_data


## Site definitions and metadata

### Step 32

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
# calculate the average surface pressure for all the 4 sites only for the winter months of November, December, January, February
hanle_avg_p = surface_pressure_data['sp'].interp(latitude = 32.7789, longitude = 78.9650).where(surface_pressure_data['time.month'].isin([1,2,11,12])).median()
merak_avg_p = surface_pressure_data['sp'].interp(latitude = 33.7828, longitude = 78.5778).where(surface_pressure_data['time.month'].isin([1,2,11,12])).median()

site_a_avg_p = surface_pressure_data['sp'].interp(latitude = 34.25, longitude = 78.75).where(surface_pressure_data['time.month'].isin([1,2,11,12])).median()
site_b_avg_p = surface_pressure_data['sp'].interp(latitude = 33, longitude = 78).where(surface_pressure_data['time.month'].isin([1,2,11,12])).median()

print(hanle_avg_p, merak_avg_p, site_a_avg_p, site_b_avg_p)


<xarray.DataArray 'sp' ()> Size: 8B
array(55911.44776688)
Coordinates:
    latitude   float64 8B 32.78
    longitude  float64 8B 78.97 <xarray.DataArray 'sp' ()> Size: 8B
array(54566.69724206)
Coordinates:
    latitude   float64 8B 33.78
    longitude  float64 8B 78.58 <xarray.DataArray 'sp' ()> Size: 8B
array(52137.35685665)
Coordinates:
    latitude   float64 8B 34.25
    longitude  float64 8B 78.75 <xarray.DataArray 'sp' ()> Size: 8B
array(53741.90927287)
Coordinates:
    latitude   int64 8B 33
    longitude  int64 8B 78

## Mapping and visualization

### Step 33

This cell interpolates gridded data to site coordinates or target locations so the analysis can be performed at specific observatory positions.

In [ ]:
hanle_p = surface_pressure_data['sp'].interp(latitude = 32.7789, longitude = 78.9650)

# plot the surface pressure for Hanle
plt.plot(hanle_p.time, hanle_p)
plt.xlabel('Time')
plt.ylabel('Surface Pressure')
plt.title('Surface Pressure for Hanle')
plt.tight_layout()
plt.show()


## Site definitions and metadata

### Step 34

This cell defines reusable helper function(s) `O3_mmr_to_vmr`, `QV_to_H2O`, `Layername`, `Layercreation` so later sections can apply the same processing logic consistently.

In [ ]:
import numpy as np
import xarray as xr
from scipy.interpolate import PchipInterpolator
import os
import subprocess

# Fetch the levels in the data
data = xr.open_dataset('sites/hanle/hanle_data_100.nc')
levels = data['level'].values
levels = [int(level) for level in levels]
# Sort the levels in ascending order
sorted_levels = sorted(levels)
levels = sorted_levels

print(levels)

# Save the levels in an am_level.txt file in separate lines
with open('am_levels.txt', 'w') as f:
    for item in levels:
        f.write("%s\n" % item)

# Conversion functions
def O3_mmr_to_vmr(o3_mmr):
    o3_vmr = 1e-6*(1e6 * (28.964 / 47.9982) * o3_mmr)
    return o3_vmr

def QV_to_H2O(qv_mmr):
    h2o = qv_mmr / 0.622
    return h2o

# Layer name definition (for am)
def Layername(pressure):
    if pressure < 1:
        return 'mesosphere'
    elif pressure >= 1 and pressure < 100:
        return 'stratosphere'
    else:
        return 'troposphere'

# Layer creation
def Layercreation(line):
    layer = {}
    layer['Name'] = Layername(line[0])
    layer['Pressure'] = str(line[0])
    layer['Temperature'] = str(round(line[1], 2))
    if line[0] <= 1:
        layer['Lineshape'] = 'Voigt-Kielkopf'
    layer['Ozone'] = '{:.3e}'.format(line[2])
    layer['PWV'] = '{:.3e}'.format(line[3])
    return layer

# File writing
def FileWriting(data, filename, pressure, preamble, keyword):
    with open('sites/'+keyword+'/am/'+filename+'.amc', 'w') as am_file:
        preamble_text = ''.join(preamble).format(
            metadata['STARTFREQ'], metadata['MEASURE'],metadata['ENDFREQ'], metadata['MEASURE'],
            metadata['STEPFREQ'],  metadata['MEASURE'],metadata['ZENITH'],metadata['DEG'],
            metadata['NSCALE'])
        #am_file.write(''.join(preamble)+'\n')
        am_file.write(preamble_text + '\n')
        for n in range(data.shape[0]):
            line = [pressure[n]] + data[n].tolist()
            layer = Layercreation(line)
            section = ['layer '+layer['Name']]
            section += ['Pbase '+layer['Pressure']+' mbar']
            section += ['Tbase '+layer['Temperature']+' K']
            if 'Lineshape' in layer:
                section += ['lineshape '+layer['Lineshape']]
            section += ['column dry_air vmr']
            section += ['column h2o vmr '+layer['PWV']]
            section += ['column o3 vmr '+ layer['Ozone']]
            am_file.write('\n'.join(section)+'\n\n')
    print('\rFile written: '+filename+'.amc', end='\r', flush=True)

# Read metadata
def Metadata(keyword=None):
    if keyword is None:
        keyword = input('Enter location keyword: ')
    metadata = {}
    with open('program/metadata/'+keyword+'_metadata.txt') as f:
        for line in f:
            (key, val) = line.split('=')
            if '\n' in val:
                val = val[:-1]
            metadata[key] = val
    metadata['COORDINATES'] = tuple(map(float, metadata['COORDINATES'].split(',')))
    metadata['SURFACEPRES'] = float(metadata['SURFACEPRES'])
    if not metadata['SHORTNAME'] in os.listdir('sites'):
        os.mkdir('sites/'+metadata['SHORTNAME'])
        print('\nDirectory created:', metadata['SHORTNAME'])
    print('\nSite:\n'+metadata['NAME'], end='\n\n')
    return metadata


### Step 35

This cell carries out the next step in the workflow and has been kept in place so the notebook remains reproducible.

In [ ]:

metadata = Metadata()


## Analysis workflow

### Step 36

This cell displays the current value of an in-memory object so the notebook user can inspect its contents before proceeding.

In [ ]:
metadata


## Project setup and imports

### Step 37

This cell defines reusable helper function(s) `interpolate_to_threshold` so later sections can apply the same processing logic consistently.

In [ ]:
import os
import numpy as np
import xarray as xr
from scipy.interpolate import PchipInterpolator
import subprocess

 
# Define constants and preamble
flairs = ['o3', 't', 'q']
quarters = ['0.25', '0.5', '0.75']
preamble = open('program/preamble_valeria.txt', 'r').readlines()
metadata = Metadata()
key_num = input('Define number of data points: ')
if 'am' not in os.listdir('sites/'+metadata['SHORTNAME']):
    os.mkdir(f'sites/{metadata["SHORTNAME"]}/am')
if 'am_output' not in os.listdir('sites/'+metadata['SHORTNAME']):
    os.mkdir(f'sites/{metadata["SHORTNAME"]}/am_output')

# Interpolate to the threshold pressure level
threshold_pressure = metadata['SURFACEPRES']
dataset_interp = xr.open_dataset(f'sites/{"/".join((metadata["SHORTNAME"],metadata["SHORTNAME"]))}_data_{key_num}.nc')  
levs = np.array([i.data for i in dataset_interp.variables['level'][:]])
print("Levels in the dataset:", levs)
levs = np.sort(levs)  # Sort the levels in ascending order
levs = np.array(levs, dtype=int)  # Convert to integer type
print("Sorted levels:", levs)
am = os.listdir('sites/'+metadata['SHORTNAME']+'/am')
start = metadata['STARTFREQ']+' '+metadata['MEASURE']
end = metadata['ENDFREQ']+' '+metadata['MEASURE']
df = metadata['STEPFREQ']+' '+metadata['MEASURE']
zenith = metadata['ZENITH']+' deg'
Nscale = metadata['NSCALE']
command = ' '.join([start,end,df,zenith,Nscale])

# Interpolation function using PCHIP
def interpolate_to_threshold(variable, levels, threshold):
    interpolator = PchipInterpolator(levels, variable)
    return interpolator(threshold)

# Filter and interpolate layers
pressure = dataset_interp.variables['level'][:].data
pressure = np.array(pressure, dtype=int)  # Convert to integer type
pressure = np.sort(pressure)  # Sort the pressure levels in ascending order
print("Pressure levels in the dataset:", pressure)
percentiles = {}
for quarter in quarters:
    to_dict = {}
    for flair in flairs:
        data_mean = []
        for i in range(levs.size):
            data = np.array(dataset_interp.variables[flair][:, i])
            p = np.linspace(0, 1, data.size)
            data.sort()
            f = lambda x: np.interp(x, p, data)
            data_mean.append(f(float(quarter)))
        to_dict[flair] = np.array(data_mean)
    percentiles[quarter] = to_dict




for q in np.arange(0.25, 1, 0.25):
    to_sort = np.array([percentiles[str(q)][value] for value in ['t', 'o3', 'q']])
    to_sort = np.array([to_sort[0], O3_mmr_to_vmr(to_sort[1]), QV_to_H2O(to_sort[2])])
    below_threshold = to_sort[:, pressure < threshold_pressure]
    if below_threshold.size == 0:
        continue  # Skip if no values are below the threshold
    # Add the interpolated layer
    print("to sort", to_sort)
    print("below threshold", below_threshold)
    print("pressure", pressure)
    interpolated_layer = [interpolate_to_threshold(var, pressure, threshold_pressure) for var in to_sort]
    to_file = np.hstack((below_threshold, np.array(interpolated_layer).reshape(-1, 1)))
    filename = f'{metadata["SHORTNAME"]}_{key_num}_{str(int(q*100))}'
    FileWriting(to_file.T, filename, np.append(pressure[pressure < threshold_pressure], threshold_pressure), preamble, metadata['SHORTNAME'])

# Run the am model
am_files = np.array(os.listdir('sites/'+metadata['SHORTNAME']+'/am'))[[metadata['SHORTNAME'] in a for a in np.array(os.listdir('sites/'+metadata['SHORTNAME']+'/am'))]]
am_files = am_files[[key_num in i for i in am_files]]
for file in am_files:
    print('\rFile to model: '+file, end='\r', flush=True)
    am_input = 'am sites/'+metadata['SHORTNAME']+'/am/'+file+' '+command
    values = subprocess.run(am_input, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True, shell=True)
    with open('sites/'+metadata['SHORTNAME']+'/am_output/'+file[:-4]+'.out', 'w') as txt_file:
        txt_file.write(values.stdout)
    with open('sites/'+metadata['SHORTNAME']+'/am_output/'+file[:-4]+'_stderr.out', 'w') as txt_file:
        txt_file.write(values.stderr)


## Running the am config files

## Exports and file generation

### Step 38

This cell defines reusable helper function(s) `run_am_simulations` so later sections can apply the same processing logic consistently.

In [ ]:
import pathlib
import subprocess
import sys

def run_am_simulations():
    """
    Finds and runs 'am' simulations for all sites in the current directory.

    This script iterates through all subdirectories in a 'sites' folder.
    For each directory (assumed to be a site like 'hanle', 'merak', etc.),
    it looks for an 'am' subfolder containing '.amc' configuration files.

    It then runs the 'am' command on each '.amc' file and saves the
    standard output and standard error to an 'am_output' directory, located
    as a sibling to the 'am' directory.
    """
    # Assumes the script is run from a directory that contains a 'sites' folder.
    sites_base_dir = pathlib.Path('./sites')
    print(f"Starting simulations from base directory: {sites_base_dir.resolve()}")

    if not sites_base_dir.is_dir():
        print(f"Error: The './sites' directory was not found. Please ensure it exists and you are running the script from its parent directory.")
        return
        
    site_dirs = [d for d in sites_base_dir.iterdir() if d.is_dir() and not d.name.startswith('.')]

    if not site_dirs:
        print(f"Error: No site directories found in '{sites_base_dir}'.")
        return

    for site_dir in site_dirs:
        print(f"\\nProcessing site: {site_dir.name}")
        am_dir = site_dir / 'am'
        
        if not am_dir.is_dir():
            print(f"  - Skipping: 'am' directory not found in '{site_dir}'.")
            continue

        # Find all .amc configuration files
        amc_files = list(am_dir.glob('*.amc'))
        
        if not amc_files:
            print(f"  - Skipping: No '.amc' files found in '{am_dir}'.")
            continue
            
        # --- CORRECTED LINE ---
        # Create the output directory directly inside the site folder,
        # making it a sibling to the 'am' directory.
        output_dir = site_dir / 'am_output'
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"  - Output will be saved to: '{output_dir}'")

        # Process each .amc file
        for amc_file in amc_files:
            base_name = amc_file.stem
            print(f"    - Running configuration: {amc_file.name}")

            # Define the full paths for the output files
            output_file_path = output_dir / f"{base_name}.out"
            stderr_file_path = output_dir / f"{base_name}.stderr.out"
            
            # Construct the command to be executed, passing the filename directly
            command = ['am', str(amc_file)]

            try:
                # Execute the 'am' command using subprocess, capturing its output
                result = subprocess.run(
                    command,
                    capture_output=True,
                    text=True,
                    check=False
                )

                # Write the captured standard output to the .out file
                with open(output_file_path, 'w') as f_out:
                    f_out.write(result.stdout)
                
                # Write the captured standard error to the .stderr.out file
                with open(stderr_file_path, 'w') as f_err:
                    f_err.write(result.stderr)
                
                # Check if the 'am' command reported an error
                if result.returncode != 0:
                    print(f"      ! Warning: 'am' exited with code {result.returncode} for {amc_file.name}.")
                    print(f"        Stderr log saved to {stderr_file_path.name}")
                else:
                    print(f"      - Success. Output saved.")

            except FileNotFoundError:
                print("\\nFATAL ERROR: The 'am' command was not found.")
                print("Please ensure that the 'am' executable is installed and in your system's PATH.")
                sys.exit(1)
            except Exception as e:
                print(f"      ! An unexpected error occurred while processing {amc_file.name}: {e}")

    print("\\nAll simulations complete.")

if __name__ == "__main__":
    run_am_simulations()


## Plotting and analysis

## Mapping and visualization

### Step 39

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
# plot all the 4 transmittance data for all the sites in a single plot for 25th percentile (100 file)
percentiles = ['25', '50', '75']
type = ['100','1000']
files = ['hanle_'+type[0]+'_'+percentiles[0]+'.out', 
         'merak_'+type[0]+'_'+percentiles[0]+'.out', 
         'site_a_'+type[0]+'_'+percentiles[0]+'.out', 
         'site_b_'+type[0]+'_'+percentiles[0]+'.out']
names = ['hanle', 'merak', 'site_a', 'site_b']
color = ['blue', 'red', 'green', 'orange']
plt.figure(figsize=(10, 6))
for i in range(4):
    data = np.loadtxt('sites/'+names[i]+'/am_output/'+files[i])
    plt.plot(data[:, 0], data[:, 2], label=names[i], alpha=0.9, color=color[i])
plt.xlabel('Wavelength (nm)')
plt.ylabel('Transmittance')
plt.legend()
plt.title('Transmittance vs Wavelength 25th percentile')
plt.tight_layout()
plt.show()


### Step 40

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
# plot the 50th percentile of 100 and 1000 file for all the sites in a single plot
files_100 = ['hanle_100_50.out', 'merak_100_50.out', 'site_a_100_50.out', 'site_b_100_50.out']
files_1000 = ['hanle_1000_50.out', 'merak_1000_50.out', 'site_a_1000_50.out', 'site_b_1000_50.out'] 

files_100 = ['hanle_100_50.out']
files_1000 = ['hanle_1000_50.out']

plt.figure(figsize=(10, 6))

for i in range(1):
    data = np.loadtxt('sites/'+names[i]+'/am_output/'+files_100[i])
    plt.plot(data[:, 0], data[:, 3], label=names[i]+'_100')
    data = np.loadtxt('sites/'+names[i]+'/am_output/'+files_1000[i])
    plt.plot(data[:, 0], data[:, 3], label=names[i]+'_1000')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Transmittance')
# plt.xlim(200, 400)
# plt.ylim(0.6, 1)
plt.legend()
plt.title('Transmittance vs Wavelength 50th percentile')
plt.tight_layout()
plt.show()


### Step 41

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the sites
sites = ['hanle', 'merak', 'site_a', 'site_b']

# Define the file prefixes
file_prefixes = ['100', '1000']

# Set up the plot
fig, axs = plt.subplots(2, 2, figsize=(15, 8), sharex='col', sharey='row')

# Define vibrant colors
colors = ['blue', 'orange']

# Loop over the sites and plot the data
for i, site in enumerate(sites):
    row, col = divmod(i, 2)
    
    for j, prefix in enumerate(file_prefixes):
        # Load the data for the 50th percentile
        filename = f'sites/{site}/am_output/{site}_{prefix}_50.out'
        data = np.loadtxt(filename)
        
        # Plot the data
        if prefix == '100':
            label = 'All Months - 50th percentile' if i == 0 else None
            axs[row, col].plot(data[:, 0], data[:, 2], label=label, color=colors[j], linewidth=2, alpha=0.8)
            axs[row, col].fill_between(data[:, 0], data[:, 2], color=colors[j], alpha=0.3)
        else:
            label = 'NDJF Months - 50th percentile' if i == 0 else None
            axs[row, col].plot(data[:, 0], data[:, 2], label=label, color=colors[j], linewidth=2, linestyle='--', alpha=0.8)
    
    # Add a fancy text box with the site name
    axs[row, col].text(0.85, 0.70, site.replace('_', ' ').upper(), transform=axs[row, col].transAxes,
                       fontsize=18, verticalalignment='top', horizontalalignment='right', 
                       bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='darkblue', linewidth=2))
    
    # Set labels for shared axes
    if col == 0:
        axs[row, col].set_ylabel('Transmittance', fontsize=18)
    if row == 1:
        axs[row, col].set_xlabel('Wavelength (nm)', fontsize=18)
    
    # Set limits
    axs[row, col].set_xlim(10, 1000)
    axs[row, col].set_ylim(0, 1)

    # Remove ticks from the shared y-axis
    if col == 1:
        axs[row, col].tick_params(labelleft=False)
    # Remove ticks from the shared x-axis
    if row == 0:
        axs[row, col].tick_params(labelbottom=False)

# Add legend inside the first subplot only
handles, labels = axs[0, 0].get_legend_handles_labels()
axs[0, 0].legend(handles, labels, loc='best', fontsize=14, fancybox=True, shadow=True)

# Adjust layout for publication quality
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.subplots_adjust(hspace=0, wspace=0)

# Customize overall appearance
plt.rc('font', size=18)
plt.rc('axes', titlesize=20)
plt.rc('axes', labelsize=18)
plt.rc('xtick', labelsize=16)
plt.rc('ytick', labelsize=16)
plt.rc('legend', fontsize=16)
plt.rc('figure', titlesize=22)

# Show the plot
plt.show()


### Step 42

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# 1. PLOTTING CONFIGURATION
# -----------------------------------------------------------------------------
# Use global settings for a consistent, professional, LaTeX-formatted look.
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "axes.labelsize": 28,
    "axes.titlesize": 24,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
    "legend.fontsize": 18,
    "axes.linewidth": 2.0,
})

# Define unique styles for each site for clear differentiation
sites = ['hanle', 'merak', 'site_a', 'site_b']
site_styles = {
    'hanle':  {'color': '#00429d', 'linestyle': 'solid', 'label': 'Hanle'},
    'merak':  {'color': '#93003a', 'linestyle': 'dashed', 'label': 'Merak'},
    'site_a': {'color': '#009b77', 'linestyle': 'solid', 'label': 'Site A'},
    'site_b': {'color': '#ffa600', 'linestyle': 'dashdot', 'label': 'Site B'}
}

# Define the four cases to plot
plot_cases = [
    {'title': r'All Months - $50^{\mathrm{th}}$ Percentile', 'prefix': '100', 'percentile': '50'},
    {'title': r'NDJF Months - $50^{\mathrm{th}}$ Percentile', 'prefix': '1000', 'percentile': '50'},
    {'title': r'All Months - $25^{\mathrm{th}}$ Percentile', 'prefix': '100', 'percentile': '25'},
    {'title': r'NDJF Months - $25^{\mathrm{th}}$ Percentile', 'prefix': '1000', 'percentile': '25'}
]

# 2. PLOTTING LOOP
# -----------------------------------------------------------------------------
fig, axs = plt.subplots(2, 2, figsize=(18, 14), sharex=True, sharey=True)

print("Generating restructured publication-quality transmittance plots...")

# Loop over the four cases and populate each subplot
for ax, case in zip(axs.flat, plot_cases):
    ax.set_title(case['title'])
    
    # In each subplot, loop over all sites
    for site in sites:
        filename = f"sites/{site}/am_output/{site}_{case['prefix']}_{case['percentile']}.out"
        
        try:
            data = np.loadtxt(filename)
            # Use the style defined for the current site
            style = site_styles[site]
            ax.plot(data[:, 0], data[:, 2], 
                    color=style['color'], 
                    linestyle=style['linestyle'], 
                    label=style['label'], 
                    linewidth=2.5, 
                    alpha=0.5)
        except FileNotFoundError:
            print(f"Warning: Data file not found, skipping. -> {filename}")
            continue

# 3. FINALIZATION AND COSMETICS
# -----------------------------------------------------------------------------

# --- Apply settings to all subplots ---
for i, ax in enumerate(axs.flat):
    row, col = divmod(i, 2)
    
    ax.set_xlim(10, 1000)
    ax.set_ylim(-0.02, 1.02)
    ax.minorticks_on()
    ax.tick_params(axis='both', which='major', direction='in', width=1.5, length=8, top=True, right=True)
    ax.tick_params(axis='both', which='minor', direction='in', width=1, length=4, top=True, right=True)

    # Control axis label visibility for shared axes
    if col > 0:
        ax.tick_params(labelleft=False)
    if row < 1:
        ax.tick_params(labelbottom=False)

# --- Add figure-level elements ---

# Create a single legend in the top-left plot, as requested
handles, labels = axs[0, 0].get_legend_handles_labels()
# Since we plotted all sites on all axes, the labels are duplicated. Get unique ones.
from collections import OrderedDict
unique_labels = OrderedDict(zip(labels, handles))
axs[0, 0].legend(unique_labels.values(), unique_labels.keys(), loc='lower left', frameon=True, facecolor='white', framealpha=0.7, edgecolor='black')

# Add shared X and Y labels for the entire figure
fig.text(0.5, 0.06, r'Wavelength (nm)', ha='center', va='center', fontsize=28)
fig.text(0.07, 0.5, r'Transmittance', ha='center', va='center', rotation='vertical', fontsize=28)

# Remove spacing between subplots, as requested
plt.subplots_adjust(hspace=0, wspace=0)

# --- Save and show the final plot ---
output_filename = 'Transmittance_Cases_Comparison_Publication.pdf'
plt.savefig(output_filename, dpi=400)
print(f"\n→ Plot successfully saved as {output_filename}")

plt.show()


### Step 43

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# 1. PLOTTING CONFIGURATION
# -----------------------------------------------------------------------------
# Use global settings for a consistent, professional, LaTeX-formatted look.
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "axes.labelsize": 28,
    "axes.titlesize": 24,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
    "legend.fontsize": 18,
    "axes.linewidth": 2.0,
})

# Define unique styles for each site (consistent with previous plot)
sites = ['hanle', 'merak', 'site_a', 'site_b']
site_styles = {
    'hanle':  {'color': '#00429d', 'linestyle': '-', 'label': 'Hanle'},
    'merak':  {'color': '#93003a', 'linestyle': '--', 'label': 'Merak'},
    'site_a': {'color': '#009b77', 'linestyle': ':', 'label': 'Site A'},
    'site_b': {'color': '#ffa600', 'linestyle': '-.', 'label': 'Site B'}
}

# Define the four astronomical windows to plot
# Using the 25th percentile of winter months for best-case scenario
case = {'prefix': '1000', 'percentile': '25'}
windows = [
    {'title': r'350 GHz Window', 'xlim': (320, 380)},
    {'title': r'450 GHz Window', 'xlim': (420, 510)},
    {'title': r'650 GHz Window', 'xlim': (620, 720)},
    {'title': r'850 GHz Window', 'xlim': (800, 920)}
]

# 2. PLOTTING LOOP
# -----------------------------------------------------------------------------
fig, axs = plt.subplots(2, 2, figsize=(18, 14), sharey=True)

print("Generating zoomed-in plots of THz astronomical windows...")

# Loop over the four windows and populate each subplot
for ax, window in zip(axs.flat, windows):
    ax.set_title(window['title'])
    
    # In each subplot, loop over all sites
    for site in sites:
        filename = f"sites/{site}/am_output/{site}_{case['prefix']}_{case['percentile']}.out"
        
        try:
            data = np.loadtxt(filename)
            style = site_styles[site]
            # Convert wavelength (nm) to frequency (GHz) for plotting
            # Frequency (GHz) = c (m/s) / (wavelength (nm) * 1e-9) / 1e9 = 299792.458 / wavelength_nm
            frequency_ghz = 299792.458 / data[:, 0]
            
            ax.plot(frequency_ghz, data[:, 2], 
                    color=style['color'], 
                    linestyle=style['linestyle'], 
                    label=style['label'], 
                    linewidth=2.5, 
                    alpha=0.9)
        except FileNotFoundError:
            print(f"Warning: Data file not found, skipping. -> {filename}")
            continue

# 3. FINALIZATION AND COSMETICS
# -----------------------------------------------------------------------------

# --- Apply settings to all subplots ---
for i, (ax, window) in enumerate(zip(axs.flat, windows)):
    row, col = divmod(i, 2)
    
    ax.set_xlim(window['xlim'])
    ax.set_ylim(-0.02, 1.02)
    ax.minorticks_on()
    ax.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.7)
    ax.tick_params(axis='both', which='major', direction='in', width=1.5, length=8, top=True, right=True)
    ax.tick_params(axis='both', which='minor', direction='in', width=1, length=4, top=True, right=True)

    if col > 0:
        ax.tick_params(labelleft=False)
    if row < 1:
        ax.tick_params(labelbottom=False)

# --- Add figure-level elements ---
handles, labels = axs[0, 0].get_legend_handles_labels()
from collections import OrderedDict
unique_labels = OrderedDict(zip(labels, handles))
# Place legend in a shared space, e.g., upper center of the figure
fig.legend(unique_labels.values(), unique_labels.keys(), loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=4, frameon=False, fontsize=22)

fig.text(0.5, 0.06, r'Frequency (GHz)', ha='center', va='center', fontsize=28)
fig.text(0.07, 0.5, r'Transmittance', ha='center', va='center', rotation='vertical', fontsize=28)

plt.tight_layout(rect=[0.08, 0.08, 1, 0.95]) # Adjust rect for shared labels/legend

# --- Save and show the final plot ---
output_filename = 'THz_Astronomy_Windows_Comparison.pdf'
plt.savefig(output_filename, dpi=400)
print(f"\n→ Zoomed-in plot successfully saved as {output_filename}")

plt.show()


### Step 44

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the sites
sites = ['hanle', 'merak', 'site_a', 'site_b']

# Define the file prefixes
file_prefixes = ['100', '1000']

# Setup plot with desired aesthetics
plt.rc('font', family='serif', size=18)
plt.rcParams['xtick.major.size'] = 6
plt.rcParams['xtick.minor.size'] = 4
plt.rcParams['ytick.major.size'] = 6
plt.rcParams['ytick.minor.size'] = 4

# Set up the plot
fig, axs = plt.subplots(2, 2, figsize=(20, 8), sharex='col', sharey='row')

# Define vibrant colors
colors = ['blue', 'orange']

# Loop over the sites and plot the data
for i, site in enumerate(sites):
    row, col = divmod(i, 2)
    
    for j, prefix in enumerate(file_prefixes):
        # Load the data for the 50th percentile
        filename = f'sites/{site}/am_output/{site}_{prefix}_50.out'
        data = np.loadtxt(filename)
        
        # Plot the data
        if prefix == '100':
            label = 'All Months - 50th percentile' if i == 0 else None
            axs[row, col].plot(data[:, 0], data[:, 3], label=label, color=colors[j], linewidth=2, alpha=0.8)
            #axs[row, col].fill_between(data[:, 0], data[:, 3], color=colors[j], alpha=0.3)
        else:
            label = 'NDJF Months - 50th percentile' if i == 0 else None
            axs[row, col].plot(data[:, 0], data[:, 3], label=label, color=colors[j], linewidth=2, linestyle='--', alpha=0.8)
            axs[row, col].fill_between(data[:, 0], data[:, 3], color=colors[j], alpha=0.3)

    # Add a fancy text box with the site name
    axs[row, col].text(0.95, 0.55, site.replace('_', ' ').upper(), transform=axs[row, col].transAxes,
                       fontsize=18, verticalalignment='top', horizontalalignment='right', 
                       bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='darkblue', linewidth=2))
    
    # Set limits
    axs[row, col].set_xlim(10, 1000)
    axs[row, col].set_ylim(0,280)

    # Add a dotted line at 0.8 transmittance
    #axs[row, col].axhline(y=0.8, color='gray', linestyle=':', linewidth=2.5)

    # Remove ticks from the shared y-axis
    if col == 1:
        axs[row, col].tick_params(labelleft=False)
    # Remove ticks from the shared x-axis
    if row == 0:
        axs[row, col].tick_params(labelbottom=False)
        # Remove the 0 in y-axis labels for plots in the first row
        axs[row, col].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '' if x == 0 else f'{x:.1f}'))


# Add legend inside the first subplot only
handles, labels = axs[0, 0].get_legend_handles_labels()
handles.append(axs[0, 0].lines[-1])  # Add the dotted line handle
#labels.append('Transmittance = 0.8')  # Add the dotted line label
axs[0, 0].legend(handles, labels, loc='lower right', fontsize=12, frameon=True, facecolor='white', edgecolor='black', framealpha=1, shadow=True)

# Add shared x and y labels
fig.text(0.5, 0.04, 'Frequency (ν in GHz)', ha='center', fontsize=20)
fig.text(0.04, 0.5, 'Brightness Temperature (K)', va='center', rotation='vertical', fontsize=20)

# Adjust layout for publication quality
plt.tight_layout(rect=[0.05, 0.05, 1, 0.95])
plt.subplots_adjust(hspace=0, wspace=0)

# Apply formatting to axes and plot borders
for ax in axs.flat:
    ax.set_facecolor('white')
    ax.spines['top'].set_linewidth(2)
    ax.spines['right'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)
    ax.spines['left'].set_linewidth(2)
    ax.tick_params(axis='both', direction='in', width=1.5, labelsize=16, which='both')
    ax.minorticks_on()

plt.savefig('BrightnessTemperature_vs_Frequency.pdf', dpi=600)
plt.show()


### Step 45

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────
# 1.  Matplotlib / LaTeX configuration
# ─────────────────────────────────────────────
plt.rc('font', family='serif', size=18)
plt.rc('xtick.major', size=6)
plt.rc('xtick.minor', size=4)
plt.rc('ytick.major', size=6)
plt.rc('ytick.minor', size=4)

plt.rcParams['text.usetex'] = True
plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'   # add amsmath

# ─────────────────────────────────────────────
# 2.  Paths, site list, colours
# ─────────────────────────────────────────────
sites         = ['hanle', 'merak', 'site_a', 'site_b']
file_prefixes = ['100', '1000']                 # 100 → all months, 1000 → NDJF
colours       = ['blue', 'orange']              # consistent colour order

# ─────────────────────────────────────────────
# 3.  Figure / axis grid
# ─────────────────────────────────────────────
fig, axs = plt.subplots(2, 2, figsize=(20, 8), sharex='col', sharey='row')

# ─────────────────────────────────────────────
# 4.  Main plotting loop
# ─────────────────────────────────────────────
for i, site in enumerate(sites):
    row, col = divmod(i, 2)
    ax = axs[row, col]

    for j, prefix in enumerate(file_prefixes):
        # Load transmittance file
        fname = f"sites/{site}/am_output/{site}_{prefix}_50.out"
        data  = np.loadtxt(fname)

        # Choose style & label
        if prefix == '100':   # all months
            label = 'All Months – 50th percentile' if i == 0 else None
            ax.plot(data[:, 0], data[:, 2], lw=2, alpha=0.8,
                    color=colours[j], label=label)
            ax.fill_between(data[:, 0], data[:, 2],
                            color=colours[j], alpha=0.30)
        else:                 # NDJF only
            label = 'NDJF Months – 50th percentile' if i == 0 else None
            ax.plot(data[:, 0], data[:, 2], lw=2, alpha=0.8,
                    color=colours[j], ls='--', label=label)

    # ── annotation with site name ──────────────────────────
    ax.text(0.95, 0.93, site.replace('_', ' ').upper(),
            transform=ax.transAxes, ha='right', va='top',
            fontsize=18,
            bbox=dict(boxstyle='round,pad=0.5',
                      facecolor='lightyellow',
                      edgecolor='darkblue', linewidth=2))

    # Axes limits, 0.8 reference line
    ax.set_xlim(10, 1000)
    ax.set_ylim(0, 1)
    ax.axhline(0.8, color='black', ls=':', lw=2.5)

    # Tick visibility tweaks
    if col == 1:
        ax.tick_params(labelleft=False)
    if row == 0:
        ax.tick_params(labelbottom=False)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(
            lambda y, _: '' if y == 0 else f'{y:.1f}'))

# ─────────────────────────────────────────────
# 5.  Legend (only once, in upper-left panel)
# ─────────────────────────────────────────────
handles, labels = axs[0, 0].get_legend_handles_labels()
# last handle in that axes is the 0.8 dotted line
handles.append(axs[0, 0].lines[-1])
labels.append('Transmittance = 0.8')
axs[0, 0].legend(handles, labels, loc='center right',
                 fontsize=12, frameon=True,
                 facecolor='white', edgecolor='black',
                 framealpha=1, shadow=True)

# ─────────────────────────────────────────────
# 6.  Shared x/y labels and final cosmetics
# ─────────────────────────────────────────────
fig.text(0.5, 0.04, r'Frequency ($\nu$ in GHz)', ha='center', fontsize=20)
fig.text(0.04, 0.5, 'Transmittance', va='center',
         rotation='vertical', fontsize=20)

plt.tight_layout(rect=[0.05, 0.05, 1, 0.95])
plt.subplots_adjust(hspace=0, wspace=0)

# consistent border / tick styling
for ax in axs.flat:
    ax.set_facecolor('white')
    for spine in ('top', 'right', 'bottom', 'left'):
        ax.spines[spine].set_linewidth(2)
    ax.tick_params(axis='both', direction='in', width=1.5,
                   which='both', labelsize=16)
    ax.minorticks_on()

# ─────────────────────────────────────────────
# 7.  Save & show
# ─────────────────────────────────────────────
plt.savefig('Transmittance_vs_Frequency.pdf', dpi=600)
plt.show()


### Final Plots

## Mapping and visualization

### Step 46

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ─────────────────────────────────────────────
# 1.  Matplotlib / LaTeX configuration
# ─────────────────────────────────────────────
plt.rc('font', family='serif', size=18)
plt.rc('xtick.major', size=6)
plt.rc('xtick.minor', size=4)
plt.rc('ytick.major', size=6)
plt.rc('ytick.minor', size=4)

plt.rcParams['text.usetex'] = True
plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'   # add amsmath

# ─────────────────────────────────────────────
# 2.  Paths, site list, colours
# ─────────────────────────────────────────────
sites         = ['hanle', 'merak', 'site_a', 'site_b']
file_prefixes = ['100', '1000']                 # 100 → all months, 1000 → NDJF
colours       = ['blue', 'orange']              # consistent colour order

# ─────────────────────────────────────────────
# 3.  Figure / axis grid
# ─────────────────────────────────────────────
fig, axs = plt.subplots(2, 2, figsize=(20, 8), sharex='col', sharey='row')

DISPLAY = {"hanle": "IAO-HANLE", "merak": "NLST-MERAK"}  # <— add this


# ─────────────────────────────────────────────
# 4.  Main plotting loop
# ─────────────────────────────────────────────
for i, site in enumerate(sites):
    row, col = divmod(i, 2)
    ax = axs[row, col]

    for j, prefix in enumerate(file_prefixes):
        # Load transmittance file for 50th percentile
        fname_50 = f"sites/{site}/am_output/{site}_{prefix}_50.out"
        data_50  = np.loadtxt(fname_50)

        # Choose style & label
        if prefix == '100':   # all months
            label = 'All Months: p50' if i == 0 else None
            ax.plot(data_50[:, 0], data_50[:, 2], lw=2, alpha=0.8,
                    color=colours[j], label=label)
            ax.fill_between(data_50[:, 0], data_50[:, 2],
                            color=colours[j], alpha=0.25)
        else:                 # NDJF only
            # --- Plot 50th percentile for NDJF ---
            label_50 = 'NDJF: p50' if i == 0 else None
            ax.plot(data_50[:, 0], data_50[:, 2], lw=2, alpha=0.8,
                    color='darkred', ls='solid', label=label_50)

            # --- Added code for 25th percentile ---
            fname_25 = f"sites/{site}/am_output/{site}_{prefix}_25.out"
            data_25  = np.loadtxt(fname_25)

            label_25 = 'NDJF: p25' if i == 0 else None
            ax.plot(data_25[:, 0], data_25[:, 2], lw=2, alpha=0.8,
                    color=colours[j], ls=':', label=label_25)
    
    panel_label = DISPLAY.get(site, site.replace('_', ' ').upper())
    ax.text(
        0.95, 0.93, panel_label,           # <— changed from site.replace(...).upper()
        transform=ax.transAxes,
        ha='right', va='top',
        fontsize=28, weight='bold',
        bbox=dict(
            boxstyle='square,pad=0.4',
            facecolor='lemonchiffon',
            edgecolor='black',
            linewidth=1.8,
            alpha=0.99
        )
    )
    # Axes limits, 0.8 reference line
    ax.set_xlim(10, 1000)
    ax.set_ylim(0, 1)
    ax.axhline(0.8, color='black', ls=':', lw=2.5)

    # --- Tick LABEL visibility tweaks ---
    # This part only affects the numbers, not the ticks themselves.
    if col == 1:
        ax.tick_params(labelleft=False)
    if row == 0:
        ax.tick_params(labelbottom=False)

# ─────────────────────────────────────────────
# 5.  Legend (Figure-level, at the top)
# ─────────────────────────────────────────────
handles, labels = axs[0, 0].get_legend_handles_labels()
handles.append(axs[0, 0].lines[-1])
labels.append('Transmittance = 0.8')

fig.legend(handles, labels,
           loc='upper center',
           bbox_to_anchor=(0.5, 1.01),
           ncol=4,
           fontsize=26,
           frameon=False,
           facecolor='white',
           edgecolor='black',
           framealpha=1,
         )

# ─────────────────────────────────────────────
# 6.  Shared x/y labels and final cosmetics
# ─────────────────────────────────────────────
fig.text(0.5, 0.04, r'Frequency ($\nu$ in GHz)', ha='center', fontsize=32)
fig.text(0.04, 0.5, 'Transmittance', va='center',
         rotation='vertical', fontsize=32)

plt.tight_layout(rect=[0.05, 0.05, 1, 0.95])

# --- START: Final Adjustments ---
# Introduce a very small gap between subplots
plt.subplots_adjust(hspace=0.04, wspace=0.04)

# Consistent border / tick styling for ALL subplots
for ax in axs.flat:
    ax.set_facecolor('white')
    for spine in ('top', 'right', 'bottom', 'left'):
        ax.spines[spine].set_linewidth(2)

    # Ensure ticks are drawn on ALL FOUR sides of each plot
    ax.tick_params(axis='both', direction='in', width=1.5,
                   which='both', labelsize=26, top=True, right=True)
    ax.minorticks_on()
    ax.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.1f'))
# --- END: Final Adjustments ---

# in the top subplot [0, 0], use 0, 0.25, 0.5, 0.75, 1 for y-ticks
axs[0, 0].set_yticks([0, 0.25, 0.5, 0.75, 1])
axs[0, 0].set_yticklabels(['0', '0.25', '0.50', '0.75', '1'])

# in subplt[1, 0], use 0,0.25,0.5,0.75
axs[1, 0].set_yticks([0, 0.25, 0.5, 0.75])
axs[1, 0].set_yticklabels(['0', '0.25', '0.50', '0.75']) 
# in subplt[0, 1], use 0.2, 0.4, 0.6, 0.8

# ─────────────────────────────────────────────
# 7.  Save & show
# ─────────────────────────────────────────────
plt.savefig('Transmittance_vs_Frequency_Final_Adjusted.pdf', dpi=600)
plt.show()


### Step 47

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# ─────────────────────────────────────────────
# 1.  Matplotlib / LaTeX configuration
# ─────────────────────────────────────────────
plt.rc('font', family='serif', size=18)
plt.rc('xtick.major', size=6)
plt.rc('xtick.minor', size=4)
plt.rc('ytick.major', size=6)
plt.rc('ytick.minor', size=4)

plt.rcParams['text.usetex'] = True
plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'   # add amsmath

# ─────────────────────────────────────────────
# 2.  Paths, site list, colours
# ─────────────────────────────────────────────
sites         = ['hanle', 'merak', 'site_a', 'site_b']
file_prefixes = ['100', '1000']                 # 100 → all months, 1000 → NDJF
colours       = ['blue', 'orange']              # consistent colour order

# APEX file (50th percentile only)
apex_file = '/Users/wavefunction/ASU Dropbox/Tanmay Singh/THz_Mac/apex_analysis/sites/apex/am_output/apex_four_50.out'
if not os.path.exists(apex_file):
    raise FileNotFoundError(f"APEX file not found: {apex_file}")
apex_data = np.loadtxt(apex_file)  # cols: [freq, ..., trans]

# ─────────────────────────────────────────────
# 3.  Figure / axis grid
# ─────────────────────────────────────────────
fig, axs = plt.subplots(2, 2, figsize=(20, 8), sharex='col', sharey='row')

DISPLAY = {"hanle": "IAO-HANLE", "merak": "NLST-MERAK"}

# ─────────────────────────────────────────────
# 4.  Main plotting loop
# ─────────────────────────────────────────────
for i, site in enumerate(sites):
    row, col = divmod(i, 2)
    ax = axs[row, col]

    for j, prefix in enumerate(file_prefixes):
        # Load transmittance file for 50th percentile
        fname_50 = f"sites/{site}/am_output/{site}_{prefix}_50.out"
        if not os.path.exists(fname_50):
            raise FileNotFoundError(f"Missing file: {fname_50}")
        data_50  = np.loadtxt(fname_50)

        # Choose style & label
        if prefix == '100':   # all months
            label = 'All Months: p50' if i == 0 else None
            ax.plot(
                data_50[:, 0], data_50[:, 2],
                lw=2, alpha=0.8,
                color=colours[j],
                label=label
            )
            ax.fill_between(
                data_50[:, 0], data_50[:, 2],
                color=colours[j], alpha=0.25
            )
        else:                 # NDJF only
            # 50th percentile NDJF
            label_50 = 'NDJF: p50' if i == 0 else None
            ax.plot(
                data_50[:, 0], data_50[:, 2],
                lw=2, alpha=0.8,
                color='darkred', ls='solid',
                label=label_50
            )

            # 25th percentile NDJF
            fname_25 = f"sites/{site}/am_output/{site}_{prefix}_25.out"
            if not os.path.exists(fname_25):
                raise FileNotFoundError(f"Missing file: {fname_25}")
            data_25  = np.loadtxt(fname_25)

            label_25 = 'NDJF: p25' if i == 0 else None
            ax.plot(
                data_25[:, 0], data_25[:, 2],
                lw=2, alpha=0.8,
                color=colours[j], ls=':',
                label=label_25
            )

    # APEX: same curve in every panel
    apex_label = 'APEX: p50' if i == 0 else None
    ax.plot(
        apex_data[:, 0], apex_data[:, 2],
        lw=2, alpha=0.6,
        color='black', ls='-.',
        label=apex_label,
        zorder = 0
    )

    # Panel label
    panel_label = DISPLAY.get(site, site.replace('_', ' ').upper())
    ax.text(
        0.95, 0.93, panel_label,
        transform=ax.transAxes,
        ha='right', va='top',
        fontsize=28, weight='bold',
        bbox=dict(
            boxstyle='square,pad=0.4',
            facecolor='lemonchiffon',
            edgecolor='black',
            linewidth=1.8,
            alpha=0.99
        )
    )

    # Axes limits, 0.8 reference line
    ax.set_xlim(10, 1000)
    ax.set_ylim(0, 1)
    ax.axhline(0.8, color='black', ls=':', lw=2.5)

    # Tick LABEL visibility tweaks (not the ticks themselves)
    if col == 1:
        ax.tick_params(labelleft=False)
    if row == 0:
        ax.tick_params(labelbottom=False)

# ─────────────────────────────────────────────
# 5.  Legend (Figure-level, at the top)
# ─────────────────────────────────────────────
handles, labels = axs[0, 0].get_legend_handles_labels()
# Add 0.8 reference line
handles.append(axs[0, 0].lines[-1])
labels.append('Transmittance = 0.8')

fig.legend(
    handles, labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.01),
    ncol=5,                 # now 5 items: All, NDJF p50, NDJF p25, APEX, 0.8 line
    fontsize=26,
    frameon=False,
    facecolor='white',
    edgecolor='black',
    framealpha=1,
)

# ─────────────────────────────────────────────
# 6.  Shared x/y labels and final cosmetics
# ─────────────────────────────────────────────
fig.text(0.5, 0.04, r'Frequency ($\nu$ in GHz)', ha='center', fontsize=32)
fig.text(0.03, 0.5, 'Transmittance', va='center',
         rotation='vertical', fontsize=32)

plt.tight_layout(rect=[0.05, 0.05, 1, 0.95])

# --- START: Final Adjustments ---
plt.subplots_adjust(hspace=0.04, wspace=0.04)

for ax in axs.flat:
    ax.set_facecolor('white')
    for spine in ('top', 'right', 'bottom', 'left'):
        spine_obj = ax.spines[spine]
        spine_obj.set_linewidth(2)

    ax.tick_params(
        axis='both', direction='in', width=1.5,
        which='both', labelsize=26, top=True, right=True
    )
    ax.minorticks_on()
    ax.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.1f'))
# --- END: Final Adjustments ---

# Custom y-ticks per panel where specified
# [0, 0]: 0, 0.25, 0.5, 0.75, 1
axs[0, 0].set_yticks([0, 0.25, 0.5, 0.75, 1])
axs[0, 0].set_yticklabels(['0', '0.25', '0.50', '0.75', '1'])

# [1, 0]: 0, 0.25, 0.5, 0.75
axs[1, 0].set_yticks([0, 0.20, 0.40, 0.60, 0.80,1.0])
axs[1, 0].set_yticklabels(['0', '0.20', '0.40', '0.60', '0.80', '1.00'])

# [0, 1]: 0.2, 0.4, 0.6, 0.8
axs[0, 1].set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
axs[0, 1].set_yticklabels(['0.20', '0.40', '0.60', '0.80', '1.00'])

# ─────────────────────────────────────────────
# 7.  Save & show
# ─────────────────────────────────────────────
plt.savefig('Transmittance_vs_Frequency_Final_Adjusted_with_APEX.pdf', dpi=600)
plt.show()


## Helper functions

### Step 48

This cell defines reusable helper function(s) `max_freq_ge_thresh`, `fmt` so later sections can apply the same processing logic consistently.

In [ ]:
threshold = 0.8

def max_freq_ge_thresh(path, thr=threshold, col_freq=0, col_T=2):
    try:
        arr = np.loadtxt(path)
    except OSError:
        return None

    if arr.ndim == 1 or arr.size == 0:
        return None

    mask = arr[:, col_T] >= thr
    if np.any(mask):
        return float(arr[mask, col_freq].max())
    return None


results = {s: {"all_p50": None, "ndjf_p50": None, "ndjf_p25": None} for s in sites}

# ---------- SITE LOOP ----------
for site in sites:
    f_all_p50  = f"sites/{site}/am_output/{site}_100_50.out"
    f_ndjf_p50 = f"sites/{site}/am_output/{site}_1000_50.out"
    f_ndjf_p25 = f"sites/{site}/am_output/{site}_1000_25.out"

    results[site]["all_p50"]  = max_freq_ge_thresh(f_all_p50)
    results[site]["ndjf_p50"] = max_freq_ge_thresh(f_ndjf_p50)
    results[site]["ndjf_p25"] = max_freq_ge_thresh(f_ndjf_p25)


# ---------- APEX ----------
apex_result = max_freq_ge_thresh(apex_file)

# ---------- PRINT ----------
print("\nMax frequency with Transmittance ≥ 0.8 (GHz)")
print("".ljust(12) + "All p50".rjust(12) + "NDJF p50".rjust(12) + "NDJF p25".rjust(12))
print("-" * 48)

def fmt(x): return f"{x:.2f}" if x is not None else "—"

for site in sites:
    a = results[site]["all_p50"]
    n50 = results[site]["ndjf_p50"]
    n25 = results[site]["ndjf_p25"]
    print(f"{site.upper():<12}{fmt(a):>12}{fmt(n50):>12}{fmt(n25):>12}")

print("-" * 48)
print(f"{'APEX':<12}{fmt(apex_result):>12}{'—':>12}{'—':>12}")


# ---------- SAVE CSV ----------
with open("max_freq_ge_0p8_per_curve.csv", "w") as f:
    f.write("site,all_p50,ndjf_p50,ndjf_p25\n")
    for site in sites:
        a = results[site]["all_p50"]
        n50 = results[site]["ndjf_p50"]
        n25 = results[site]["ndjf_p25"]
        f.write(f"{site},{'' if a is None else a},{'' if n50 is None else n50},{'' if n25 is None else n25}\n")

    # APEX appended
    f.write(f"APEX,{apex_result},,\n")

print("\nWrote: max_freq_ge_0p8_per_curve.csv")


Max frequency with Transmittance ≥ 0.8 (GHz)
                 All p50    NDJF p50    NDJF p25
------------------------------------------------
HANLE             290.18      312.77      314.43
MERAK             271.75      308.15      310.93
SITE_A            300.38      315.30      344.65
SITE_B            285.20      313.05      314.82

## Summary statistics and reporting

### Step 49

This cell defines reusable helper function(s) `band_avg`, `fmt` so later sections can apply the same processing logic consistently.

In [ ]:
# =========================================================
# ALMA + sub-mm atmospheric bands for comparison
# =========================================================

bands = [
    ("ALMA_B6",   211, 275),
    ("ALMA_B7",   275, 373),
    ("ALMA_B8",   385, 500),
    ("ALMA_B9",   602, 720),
    ("ALMA_B10",  787, 950),
    ("850um",     330, 360),
    ("450um",     600, 700),
    ("350um",     800, 900),
]

curve_suffixes = {
    "all_p50":  "100_50",
    "ndjf_p50": "1000_50",
    "ndjf_p25": "1000_25",
}

curve_display = {
    "all_p50":  "All_p50",
    "ndjf_p50": "NDJF_p50",
    "ndjf_p25": "NDJF_p25",
}

def band_avg(arr, nu_min, nu_max, col_freq=0, col_T=2):
    mask = (arr[:, col_freq] >= nu_min) & (arr[:, col_freq] <= nu_max)
    if not np.any(mask):
        return None
    return float(np.mean(arr[mask, col_T]))


band_results = {}

# ---------- Sites ----------
for site in sites:
    band_results[site] = {}

    for curve, suffix in curve_suffixes.items():
        path = f"sites/{site}/am_output/{site}_{suffix}.out"
        try:
            arr = np.loadtxt(path)
        except:
            continue

        band_results[site][curve] = {}

        for name, nu1, nu2 in bands:
            band_results[site][curve][name] = band_avg(arr, nu1, nu2)

# ---------- APEX ----------
band_results["APEX"] = {}
apex_arr = np.loadtxt(apex_file)
band_results["APEX"]["p50"] = {}

for name, nu1, nu2 in bands:
    band_results["APEX"]["p50"][name] = band_avg(apex_arr, nu1, nu2)

# =========================================================
# PRINT TABLE
# =========================================================

def fmt(x):
    return f"{x:.3f}" if x is not None else "—"

print("\nBand-averaged atmospheric transmittance")
print("="*120)

header = ["Site", "Curve"] + [b[0] for b in bands]
print("".join(f"{h:>15}" for h in header))
print("-"*120)

for site in band_results:
    for curve in band_results[site]:
        line = f"{site:>15}{curve_display.get(curve, curve):>15}"
        for band in [b[0] for b in bands]:
            val = band_results[site][curve].get(band)
            line += f"{fmt(val):>15}"
        print(line)

# =========================================================
# SAVE TO CSV
# =========================================================

with open("band_avg_transmittance_ALMA.csv", "w") as f:
    f.write("site,curve," + ",".join([b[0] for b in bands]) + "\n")
    for site in band_results:
        for curve in band_results[site]:
            row = [site, curve_display.get(curve, curve)]
            for band in [b[0] for b in bands]:
                val = band_results[site][curve].get(band)
                row.append("" if val is None else f"{val:.5f}")
            f.write(",".join(row) + "\n")

print("\nSaved: band_avg_transmittance_ALMA.csv")


## Mapping and visualization

### Step 50

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ─────────────────────────────────────────────
# 1.  Matplotlib / LaTeX configuration (from reference)
# ─────────────────────────────────────────────
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "axes.labelsize": 32,
    "xtick.labelsize": 26,
    "ytick.labelsize": 26,
    "legend.fontsize": 26,
    "axes.linewidth": 2,
})

# ─────────────────────────────────────────────
# 2.  Paths, site list, colours
# ─────────────────────────────────────────────
sites         = ['hanle', 'merak', 'site_a', 'site_b']
file_prefixes = ['100', '1000']
# Define a 3-color scheme for the three data series
colours       = {'all_p50': 'blue', 'ndjf_p50': 'darkred', 'ndjf_p25': 'orange'}

# ─────────────────────────────────────────────
# 3.  Figure / axis grid
# ─────────────────────────────────────────────
fig, axs = plt.subplots(2, 2, figsize=(20, 8), sharex='col', sharey='row')

# ─────────────────────────────────────────────
# 4.  Main plotting loop
# ─────────────────────────────────────────────
for i, site in enumerate(sites):
    row, col = divmod(i, 2)
    ax = axs[row, col]

    for j, prefix in enumerate(file_prefixes):
        fname_50 = f"sites/{site}/am_output/{site}_{prefix}_50.out"
        data_50  = np.loadtxt(fname_50)

        # Plotting Brightness Temperature (Column 3)
        if prefix == '100':   # all months
            label = 'All Months: p50' if i == 0 else None
            ax.plot(data_50[:, 0], data_50[:, 3], lw=2, alpha=0.8,
                    color=colours['all_p50'], label=label)
        else:                 # NDJF only
            # --- Plot 50th percentile for NDJF ---
            label_50 = 'NDJF: p50' if i == 0 else None
            ax.plot(data_50[:, 0], data_50[:, 3], lw=2, alpha=0.8,
                    color=colours['ndjf_p50'], ls='solid', label=label_50)
            ax.fill_between(data_50[:, 0], data_50[:, 3],
                            color=colours['ndjf_p50'], alpha=0.25)

            # --- Plot 25th percentile for NDJF ---
            fname_25 = f"sites/{site}/am_output/{site}_{prefix}_25.out"
            data_25  = np.loadtxt(fname_25)
            label_25 = 'NDJF: p25' if i == 0 else None
            ax.plot(data_25[:, 0], data_25[:, 3], lw=2, alpha=0.8,
                    color=colours['ndjf_p25'], ls=':', label=label_25)

    panel_label = DISPLAY.get(site, site.replace('_', ' ').upper())
    ax.text(
        0.95, 0.23, panel_label,           # <— changed from site.replace(...).upper()
        transform=ax.transAxes,
        ha='right', va='top',
        fontsize=28, weight='bold',
        bbox=dict(
            boxstyle='square,pad=0.4',
            facecolor='lemonchiffon',
            edgecolor='black',
            linewidth=1.8,
            alpha=0.99
        )
    )



    # Set axes limits for Brightness Temperature
    ax.set_xlim(10, 1000)
    ax.set_ylim(0, 280)

    # --- Tick LABEL visibility tweaks ---
    if col > 0:
        ax.tick_params(labelleft=False)
    if row < 1:
        ax.tick_params(labelbottom=False)

# ─────────────────────────────────────────────
# 5.  Legend (Figure-level, at the top)
# ─────────────────────────────────────────────
handles, labels = axs[0, 0].get_legend_handles_labels()
fig.legend(handles, labels,
           loc='upper center',
           bbox_to_anchor=(0.5, 1.03),
           ncol=3, # 3 columns for the 3 data series
           fontsize=26,
           frameon=False,
           facecolor='white',
           edgecolor='black',
           framealpha=1,
         )

# ─────────────────────────────────────────────
# 6.  Shared x/y labels and final cosmetics
# ─────────────────────────────────────────────
fig.text(0.5, 0.04, r'Frequency ($\nu$ in GHz)', ha='center', fontsize=32)
fig.text(0.04, 0.5, 'Brightness Temperature (K)', va='center',
         rotation='vertical', fontsize=32)

plt.tight_layout(rect=[0.05, 0.05, 1, 0.95])
plt.subplots_adjust(hspace=0.04, wspace=0.04)

# --- Consistent border / tick styling for ALL subplots ---
for ax in axs.flat:
    ax.set_facecolor('white')
    for spine in ('top', 'right', 'bottom', 'left'):
        ax.spines[spine].set_linewidth(2)
    # Ensure ticks are drawn on ALL FOUR sides of each plot
    ax.tick_params(axis='both', direction='in', width=1.5,
                   which='both', labelsize=26, top=True, right=True)
    ax.minorticks_on()

# --- Apply specific Y-tick formatting as per reference ---
axs[0, 0].set_yticks([0, 50, 100, 150, 200, 250])
axs[1, 0].set_yticks([0, 50, 100, 150, 200, 250])
axs[0, 1].set_yticks([0, 50, 100, 150, 200, 250])
axs[1, 1].set_yticks([0, 50, 100, 150, 200, 250])

# ─────────────────────────────────────────────
# 7.  Save & show
# ─────────────────────────────────────────────

plt.savefig('BrightnessTemperature_vs_Frequency_Publication.pdf', dpi=600)
plt.show()


### Step 51

This cell produces a figure to visualize the atmospheric variables, site comparison metrics, or derived PWV behavior.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ─────────────────────────────────────────────
# 1.  Matplotlib / LaTeX configuration
# ─────────────────────────────────────────────
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "axes.labelsize": 32,
    "xtick.labelsize": 26,
    "ytick.labelsize": 26,
    "legend.fontsize": 26,
    "axes.linewidth": 2,
})

# ─────────────────────────────────────────────
# 2.  Paths, site list, colours
# ─────────────────────────────────────────────
sites         = ['hanle', 'merak', 'site_a', 'site_b']
file_prefixes = ['100', '1000']   # 100 → all months, 1000 → NDJF

# 3-colour scheme for the three “local” series
colours = {
    'all_p50' : 'blue',
    'ndjf_p50': 'darkred',
    'ndjf_p25': 'orange',
}

# APEX file (50th percentile only) – brightness temperature column is index 3
apex_file = '/Users/wavefunction/ASU Dropbox/Tanmay Singh/THz_Mac/apex_analysis/sites/apex/am_output/apex_four_50.out'
if not os.path.exists(apex_file):
    raise FileNotFoundError(f"APEX file not found: {apex_file}")
apex_data = np.loadtxt(apex_file)

# Panel display labels
DISPLAY = {"hanle": "IAO-HANLE", "merak": "NLST-MERAK"}

# ─────────────────────────────────────────────
# 3.  Figure / axis grid
# ─────────────────────────────────────────────
fig, axs = plt.subplots(2, 2, figsize=(20, 8), sharex='col', sharey='row')

# ─────────────────────────────────────────────
# 4.  Main plotting loop
# ─────────────────────────────────────────────
for i, site in enumerate(sites):
    row, col = divmod(i, 2)
    ax = axs[row, col]

    # Local sites: ALL months + NDJF, brightness temperature
    for prefix in file_prefixes:
        fname_50 = f"sites/{site}/am_output/{site}_{prefix}_50.out"
        if not os.path.exists(fname_50):
            raise FileNotFoundError(f"Missing file: {fname_50}")
        data_50 = np.loadtxt(fname_50)

        if prefix == '100':   # all months: p50
            label = 'All Months: p50' if i == 0 else None
            ax.plot(
                data_50[:, 0],  # frequency [GHz]
                data_50[:, 3],  # brightness temperature [K]
                lw=2, alpha=0.8,
                color=colours['all_p50'],
                label=label
            )

        else:                 # NDJF only
            # 50th percentile NDJF
            label_50 = 'NDJF: p50' if i == 0 else None
            ax.plot(
                data_50[:, 0], data_50[:, 3],
                lw=2, alpha=0.8,
                color=colours['ndjf_p50'], ls='solid',
                label=label_50
            )
            ax.fill_between(
                data_50[:, 0], data_50[:, 3],
                color=colours['ndjf_p50'], alpha=0.25
            )

            # 25th percentile NDJF
            fname_25 = f"sites/{site}/am_output/{site}_{prefix}_25.out"
            if not os.path.exists(fname_25):
                raise FileNotFoundError(f"Missing file: {fname_25}")
            data_25 = np.loadtxt(fname_25)

            label_25 = 'NDJF: p25' if i == 0 else None
            ax.plot(
                data_25[:, 0], data_25[:, 3],
                lw=2, alpha=0.8,
                color=colours['ndjf_p25'], ls=':',
                label=label_25
            )

    # APEX: same brightness–temperature curve in every panel
    apex_label = 'APEX: p50' if i == 0 else None
    ax.plot(
        apex_data[:, 0],  # frequency
        apex_data[:, 3],  # brightness temperature
        lw=2, alpha=0.6,
        color='black', ls='-.',
        label=apex_label,
        zorder=0
    )

    # Panel label
    panel_label = DISPLAY.get(site, site.replace('_', ' ').upper())
    ax.text(
        0.95, 0.23, panel_label,
        transform=ax.transAxes,
        ha='right', va='top',
        fontsize=28, weight='bold',
        bbox=dict(
            boxstyle='square,pad=0.4',
            facecolor='lemonchiffon',
            edgecolor='black',
            linewidth=1.8,
            alpha=0.99
        )
    )

    # Axes limits for brightness temperature
    ax.set_xlim(10, 1000)
    ax.set_ylim(0, 280)

    # Tick LABEL visibility tweaks
    if col > 0:
        ax.tick_params(labelleft=False)
    if row < 1:
        ax.tick_params(labelbottom=False)

# ─────────────────────────────────────────────
# 5.  Legend (Figure-level, at the top)
# ─────────────────────────────────────────────
handles, labels = axs[0, 0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.03),
    ncol=4,          # all_p50, NDJF p50, NDJF p25, APEX
    fontsize=26,
    frameon=False,
    facecolor='white',
    edgecolor='black',
    framealpha=1,
)

# ─────────────────────────────────────────────
# 6.  Shared x/y labels and final cosmetics
# ─────────────────────────────────────────────
fig.text(0.5, 0.04, r'Frequency ($\nu$\,in\,GHz)', ha='center', fontsize=32)
fig.text(
    0.04, 0.5,
    r'Brightness Temperature (K)',
    va='center', rotation='vertical', fontsize=32
)

plt.tight_layout(rect=[0.05, 0.05, 1, 0.95])
plt.subplots_adjust(hspace=0.04, wspace=0.04)

# Consistent border / tick styling for ALL subplots
for ax in axs.flat:
    ax.set_facecolor('white')
    for spine in ('top', 'right', 'bottom', 'left'):
        ax.spines[spine].set_linewidth(2)
    ax.tick_params(
        axis='both', direction='in', width=1.5,
        which='both', labelsize=26,
        top=True, right=True
    )
    ax.minorticks_on()

# Y-ticks in all panels
for ax in axs.flat:
    ax.set_yticks([0, 50, 100, 150, 200, 250])

# ─────────────────────────────────────────────
# 7.  Save & show
# ─────────────────────────────────────────────
plt.savefig('BrightnessTemperature_vs_Frequency_with_APEX.pdf', dpi=600)
plt.show()
